In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("dissertation.experiments.csv")

print(f"Size: {len(df)}, Cols: {df.columns.tolist()}")

df["is_edit"] = df["task_id"].astype(str).str.startswith("edit_")
df["task_type"] = np.where(df["is_edit"], "Edit", "Create")
df["outcome"] = np.where(df["passed"], "Passed", df["failure_reason_category"].fillna("Unknown"))

df["model_label"] = df["model"]

overall = {
    "runs": len(df),
    "tasks": df["task_id"].nunique(),
    "models": df["model"].nunique(),
    "overall_pass_rate": df["passed"].mean()
}

print(f"Overall: {overall}")

task_type = df.groupby("task_type")["passed"].agg(["mean", "sum", "count"])
model = df.groupby("model")["passed"].agg(["mean", "sum", "count"]).sort_values("mean", ascending=False)
rag = df.groupby("rag_enabled")["passed"].agg(["mean", "sum", "count"])
rag_n = df.groupby(["rag_enabled", "rag_n"])["passed"].agg(["mean", "sum", "count"])
outcomes = df["outcome"].value_counts()
repair = df.groupby("static_repair_loops_taken")["passed"].agg(["mean", "sum", "count"]).sort_index()
task = df.groupby("task_id")["passed"].agg(["mean", "sum", "count"]).sort_values("mean", ascending=False)
percept_pass = df.groupby("passed")["percepts"].agg(["mean", "sum", "count"])
code_length_pass = df.groupby("passed")["code_length"].agg(["mean", "sum", "count"])
task_type_rag = df.groupby(["task_type", "rag_enabled"])["passed"].agg(["mean", "sum", "count"])
model_task_type = df.groupby(["model", "task_type"])["passed"].agg(["mean", "sum", "count"])
model_rag = df.groupby(["model", "rag_enabled"])["passed"].agg(["mean", "sum", "count"])



Size: 124, Cols: ['_id', 'task_id', 'prompt', 'rag_enabled', 'rag_n', 'model', 'temperature', 'top_k', 'top_p', 'seed', 'llm_timeout_seconds', 'repeat_penalty', 'repeat_last_n', 'num_predict', 'runner_timeout_seconds', 'code_length', 'percepts', 'runtime', 'passed', 'static_repair_loops_taken', 'timed_out', 'generated_code', 'code_output.exit_code', 'code_output.stdout', 'code_output.stderr', 'code_output.runtime', 'timestamp', 'experiment_type', 'failure_reason_category', 'exit_code']
Overall: {'runs': 124, 'tasks': 14, 'models': 5, 'overall_pass_rate': 0.3709677419354839}


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

outdir = Path("dissertation_graphs")
outdir.mkdir(exist_ok=True)

def save_bar(series, filename, title, ylabel, xlabel="", ylim=None, rotation=0, figsize=(8, 5), annotate=False):
    plt.figure(figsize=figsize)
    ax = series.plot(kind="bar")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xlabel(xlabel)
    plt.xticks(rotation=rotation, ha="right" if rotation else "center")
    if ylim is not None:
        plt.ylim(*ylim)

    if annotate:
        for p in ax.patches:
            h = p.get_height()
            if pd.notna(h):
                ax.annotate(
                    f"{h:.1f}",
                    (p.get_x() + p.get_width() / 2, h),
                    ha="center",
                    va="bottom",
                    fontsize=8
                )

    plt.tight_layout()
    plt.savefig(outdir / filename, dpi=220, bbox_inches="tight")
    plt.close()


outcome_order = ["Passed", "static checker", "timeout", "runtime error"]
outcome_counts = outcomes.reindex(outcome_order).fillna(0)
save_bar(
    outcome_counts,
    "01_outcomes.png",
    "Run Outcomes",
    "Number of runs",
    rotation=20
)

task_type_pass = task_type["mean"] * 100
save_bar(
    task_type_pass,
    "02_pass_rate_by_task_type.png",
    "Pass Rate by Task Type",
    "Pass rate (%)",
    ylim=(0, 100),
    annotate=True
)

model_pass = model["mean"] * 100
model_pass.index = [f"{idx}\n(n={model.loc[idx, 'count']})" for idx in model.index]
save_bar(
    model_pass,
    "03_pass_rate_by_model.png",
    "Pass Rate by Model",
    "Pass rate (%)",
    ylim=(0, 100),
    rotation=20,
    annotate=True
)

rag_pass = rag["mean"] * 100
rag_pass.index = ["No RAG" if idx is False else "RAG" for idx in rag_pass.index]
save_bar(
    rag_pass,
    "04_rag_vs_no_rag_overall.png",
    "Overall Pass Rate: RAG vs No RAG",
    "Pass rate (%)",
    ylim=(0, 100),
    annotate=True
)

task_type_rag_pivot = (task_type_rag["mean"] * 100).unstack()
task_type_rag_pivot = task_type_rag_pivot.rename(columns={False: "No RAG", True: "RAG"})
task_type_rag_pivot = task_type_rag_pivot.reindex(["Create", "Edit"])

plt.figure(figsize=(8, 5))
ax = task_type_rag_pivot.plot(kind="bar")
plt.title("Pass Rate by Task Type and Retrieval Setting")
plt.ylabel("Pass rate (%)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.ylim(0, 100)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", fontsize=8)
plt.tight_layout()
plt.savefig(outdir / "05_task_type_by_rag_setting.png", dpi=220, bbox_inches="tight")
plt.close()

rag_n_pass = (
    df[df["rag_enabled"] == True]
    .groupby("rag_n")["passed"]
    .agg(["mean", "count"])
    .sort_index()
)
series = rag_n_pass["mean"] * 100
series.index = [f"{idx}\n(n={rag_n_pass.loc[idx, 'count']})" for idx in series.index]
save_bar(
    series,
    "06_pass_rate_by_rag_n.png",
    "Pass Rate by Retrieval Depth (RAG Only)",
    "Pass rate (%)",
    xlabel="rag_n",
    ylim=(0, 100),
    annotate=True
)

repair_counts = df["static_repair_loops_taken"].value_counts().sort_index()

repair_counts.index = [str(int(x)) for x in repair_counts.index]

save_bar(
    repair_counts,
    "07_count_by_repair_loops.png",
    "Run Count by Repair Loop Count",
    "Number of runs",
    xlabel="Repair loops taken",
    annotate=True
)
model_task_type_pivot = (model_task_type["mean"] * 100).unstack()
plt.figure(figsize=(9, 5))
ax = model_task_type_pivot.plot(kind="bar")
plt.title("Pass Rate by Model and Task Type")
plt.ylabel("Pass rate (%)")
plt.xlabel("")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 100)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", fontsize=8)
plt.tight_layout()
plt.savefig(outdir / "08_model_by_task_type.png", dpi=220, bbox_inches="tight")
plt.close()

task_pass = task["mean"] * 100
task_pass.index = [f"{idx}\n(n={task.loc[idx, 'count']})" for idx in task.index]
save_bar(
    task_pass,
    "09_pass_rate_by_task.png",
    "Pass Rate by Task",
    "Pass rate (%)",
    ylim=(0, 100),
    rotation=55,
    figsize=(13, 6)
)

percepts_series = percept_pass["mean"].rename(index={False: "Failed", True: "Passed"})
percepts_series = percepts_series.reindex(["Passed", "Failed"])

save_bar(
    percepts_series,
    "10_average_percepts_pass_vs_fail.png",
    "Average Percepts: Passed vs Failed",
    "Average percept count",
    annotate=True
)

code_length_series = code_length_pass["mean"].rename(index={False: "Failed", True: "Passed"})
code_length_series = code_length_series.reindex(["Passed", "Failed"])

save_bar(
    code_length_series,
    "11_average_code_length_pass_vs_fail.png",
    "Average Code Length: Passed vs Failed",
    "Average code length",
    annotate=True
)

outcome_by_type = pd.crosstab(df["task_type"], df["outcome"])
outcome_by_type = outcome_by_type.reindex(index=["Create", "Edit"], columns=outcome_order).fillna(0)

plt.figure(figsize=(9, 5))
outcome_by_type.plot(kind="bar", stacked=True)
plt.title("Outcome Distribution by Task Type")
plt.ylabel("Number of runs")
plt.xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(outdir / "12_outcomes_by_task_type.png", dpi=220, bbox_inches="tight")
plt.close()

rag_type_counts = pd.crosstab(df["task_type"], df["rag_enabled"])
rag_type_counts = rag_type_counts.rename(columns={False: "No RAG", True: "RAG"})
rag_type_counts = rag_type_counts.reindex(index=["Create", "Edit"])

plt.figure(figsize=(8, 5))
rag_type_counts.plot(kind="bar", stacked=True)
plt.title("Run Count by Task Type and Retrieval Setting")
plt.ylabel("Number of runs")
plt.xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(outdir / "13_run_count_task_type_rag.png", dpi=220, bbox_inches="tight")
plt.close()

model_rag_pivot = (model_rag["mean"] * 100).unstack()
model_rag_pivot = model_rag_pivot.rename(columns={False: "No RAG", True: "RAG"})

plt.figure(figsize=(9, 5))
ax = model_rag_pivot.plot(kind="bar")
plt.title("Pass Rate by Model and Retrieval Setting")
plt.ylabel("Pass rate (%)")
plt.xlabel("")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 100)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", fontsize=8)
plt.tight_layout()
plt.savefig(outdir / "14_pass_rate_by_model_and_rag.png", dpi=220, bbox_inches="tight")
plt.close()

df["percept_bin"] = pd.cut(
    df["percepts"],
    bins=[-0.1, 0, 1, 2, 5, 100],
    labels=["0", "1", "2", "3-5", "6+"]
)

percept_bin_pass = df.groupby("percept_bin", observed=False)["passed"].mean() * 100
save_bar(
    percept_bin_pass,
    "15_success_by_percept_bin.png",
    "Success Rate by Percept Complexity",
    "Pass rate (%)",
    xlabel="Percept count bin",
    ylim=(0, 100),
    rotation=20,
    annotate=True
)

print(f"Saved graphs to: {outdir.resolve()}")
print(sorted([p.name for p in outdir.iterdir()]))

ValueError: cannot reindex on an axis with duplicate labels

<Figure size 800x500 with 0 Axes>

<Figure size 900x500 with 0 Axes>